In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats 
import warnings
import matplotlib.patches as mpatches
import os
import matplotlib.ticker as mtick
from scipy import stats 

from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import auc
from sklearn.metrics import matthews_corrcoef

from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [ ]:
datasets = ['ds1', 'ds2', 'ds3']
profiles = ['pfam', 'kegg', 'tax', 'com']
grouping = ['CB']

def getRGB(r,g,b):
    return (r/255.0, g/255.0, b/255.0)

def computeP_value(dist, score):
    kde = stats.gaussian_kde(dist)
    sns.kdeplot(x=dist)
    sa = kde.integrate_box_1d(score, np.Inf)
    plt.close()
    return sa

color_map = {'pfam':"#a81e32", 'kegg':"#002FA7", 'tax':getRGB(234,185,70), 'com':getRGB(104,148,92), "guess":getRGB(6,20,32)}
legend_properties = {'weight':'bold', 'size':20}
patch = []
for key in color_map:
    patch.append(mpatches.Patch(color=color_map[key], label=key))

In [ ]:
# make test
map_class = {
    "SR" : {"PR":1, "CR":1, "PD":0, "DD":0, "SD":0},
    "CB" : {"PR":1, "CR":1, "PD":0, "DD":0, "SD":1},
    "3C" : {"PR":2, "CR":2, "PD":0, "DD":0, "SD":1},
    "2C" : {"PR":1, "CR":1, "PD":0, "DD":0},
    "org": {"PR":2, "CR":3, "PD":0, "DD":0, "SD":1},
    "pfs": {">6":1, "<3":0},
}

def getDS(labels):
    gt = np.zeros(len(labels))
    for (i, v) in enumerate(labels):
        if v[0] == 'J':
            gt[i] = 0
        elif v[0] == 'E':
            gt[i] = 1
        elif v[0] == 'S':
            gt[i] = 2
    return gt

def getGT(labels, tag):
    gt = np.zeros(len(labels))
    for (i, v) in enumerate(labels):
        gt[i] = map_class[tag][v[-2:]]
    return gt


renameSample_test = {}
renameSample_pfs = {}
stable_list = []
with open("../data/metadata.csv") as fin:
    for line in fin:
        if not line[0] == '#':
            cont = line.strip().split(',')
            renameSample_test[cont[0]] = cont[0] + '_' +cont[5]
            if len(cont[12]) > 0:
                renameSample_pfs[cont[0]] = cont[0] + '_' +cont[12]
            if cont[5] == 'SD':
                stable_list.append(cont[0] + '_' +cont[5])
                
def loadPFAM(fname, ff_fast=False):
    df_pfam = pd.DataFrame(pd.read_csv(fname, sep='\t', index_col=0))
    return df_pfam

def loadKEGG(fname, ff_fast=False):
    df_kegg  = pd.DataFrame(pd.read_csv(fname, sep='\t', index_col=0))
    return df_kegg

def loadTAX(fname):
    df_tax  = pd.DataFrame(pd.read_csv(fname, sep=',', index_col=0))
    df_tax.sort_index(axis=1, inplace=True)
    return df_tax


df_pfam = loadPFAM(f"../data/pfam.count.tsv")
df_kegg = loadKEGG(f"../data/kegg.count.tsv")
df_tax = loadTAX(f"../data/otu.count.csv")

df_pfam.rename(columns=renameSample_test,inplace=True)
df_kegg.rename(columns=renameSample_test,inplace=True)
df_tax.rename(columns=renameSample_test,inplace=True)

df_com  = pd.concat([ df_tax, df_kegg, df_pfam]).fillna(0).copy()

map_data = {"pfam":df_pfam, "kegg":df_kegg, "tax":df_tax, "com":df_com}

training_sample = [renameSample_test[x] for x in renameSample_pfs]

In [ ]:
def my_split1(df_join, Y, ds):
    ds_label = getDS(df_join.index)
    if ds == 'dds3':
        df_train = df_join[ds_label!=2]
        y_train = Y[ds_label!=2]
        df_join = df_join[ds_label==2]
        Y = Y[ds_label==2]
        N = len(ds_label)
    elif ds in ['ds1', 'ds2', 'ds3']:
        df_join = df_join[ds_label==(int(ds[-1])-1)]
        Y = Y[ds_label==(int(ds[-1])-1)]
        df_train = pd.DataFrame()
        y_train = np.array([])
        N = len(Y)
    elif ds == 'ds12':
        df_join = df_join[ds_label!=2]
        Y = Y[ds_label!=2]
        df_train = pd.DataFrame()
        y_train = np.array([])
        N = len(Y)
    else:
        df_train = pd.DataFrame()
        y_train = np.array([])
        N = len(Y)
    rs_sp = np.random.randint(9999999)
    kf = StratifiedShuffleSplit(n_splits=1, test_size=int(np.ceil(0.3*N)), random_state=rs_sp)
    for T_index, t_index in kf.split(df_join, Y):
        df_tmp , y_tmp   = df_join.iloc[T_index], Y[T_index]
        df_test,  y_test = df_join.iloc[t_index], Y[t_index]
    df_train = pd.concat([df_train, df_tmp])
    y_train = np.concatenate((y_train, y_tmp))
    return (rs_sp, df_train, y_train, df_test, y_test)

def my_training_testing_split(df_join, ds):
    if ds in ['mix', 'dds3', 'ds1', 'ds2', 'ds3', 'ds12']:
        Y = getGT(df_join.index, sp2)
        return my_split1(df_join, Y, ds)
    if ds == 'pfs3':
        df_all = df_join.loc[training_sample]
        pfs_index = [renameSample_pfs[x[:-3]] for x in df_all.index]
        Y = getGT(pfs_index, 'pfs')
        rs_sp = np.random.randint(9999999)
        kf = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=rs_sp)
        for T_index, t_index in kf.split(df_all, Y):
            df_train, y_train = df_all.iloc[T_index], Y[T_index]
            df_test,  y_test  = df_all.iloc[t_index], Y[t_index]
        return (rs_sp, df_train, y_train, df_test, y_test)
    ds_label = getDS(df_join.index)
    df_train = df_join.loc[training_sample]
    if sp1 == 'pfs':
        pfs_index = [renameSample_pfs[x[:-3]] for x in df_train.index]
        y_train  = getGT(pfs_index, sp1)
    else:
        y_train  = getGT(df_train.index, sp1)

    if ds == 'pfs1':
        df_test  = df_join[ds_label!=2]
    elif ds == 'pfs2':
        df_test  = df_join.loc[list(set(df_join.index[ds_label!=2])-set(training_sample))]
    y_test   = getGT(df_test.index, sp2)
    return (-1, df_train, y_train, df_test, y_test)

pf = 'pfam'
sp = 'CB'
sp1 = sp2 = sp
df_guess = {ds:0 for ds in ['mix', 'ds2', 'ds3', 'dds3', 'ds12', 'pfs1', 'pfs2', 'pfs3']}
top = 100


In [ ]:
# random guess
def randomTest(y_test):
    scores = np.zeros((1000,5))
    for N in range(1000):
        p_test = np.random.randint(2, size=y_test.shape)
        scores[N,0] = recall_score(     y_test, p_test)
        scores[N,1] = precision_score(  y_test, p_test, zero_division=1)
        scores[N,2] = f1_score(         y_test, p_test)
        scores[N,3] = f1_score(         y_test, p_test, average='macro')
        scores[N,4] = matthews_corrcoef(y_test, p_test)
    df_score = pd.DataFrame(scores, columns=["recall", "precision", "f1_score", "macro_f1", "MCC"])
    return df_score


df_join = df_pfam.T.copy()
Y = getGT(df_join.index, sp)
for ds in df_guess:
    (rs_sp, df_train, y_train, df_test, y_test) = my_training_testing_split(df_join, ds)
    df_guess[ds] = randomTest(y_test)
for ds in df_guess:
    print(ds, df_guess[ds]['f1_score'].mean(), np.std(df_guess[ds]['f1_score']))
    

# Test using new matrix

In [ ]:
def computeFromPROB(df_tmp):
    probs  = df_tmp.iloc[::2].copy()
    labels = df_tmp.iloc[1::2].copy()
    scores = np.zeros((100,5))
    for N in range(100):
        prob  = probs.iloc[N]
        label = labels.iloc[N]
        y_test = (prob > 0.50)
        p_test = (label == 1)
        scores[N,0] = recall_score(     y_test, p_test)
        scores[N,1] = precision_score(  y_test, p_test, zero_division=1)
        scores[N,2] = f1_score(         y_test, p_test)
        scores[N,3] = f1_score(         y_test, p_test, average='macro')
        scores[N,4] = matthews_corrcoef(y_test, p_test)
    df_score = pd.DataFrame(scores, columns=["recall", "precision", "f1_score", "macro_f1", "MCC"])
    return df_score
        

## tax+

In [ ]:
workspace = "../data/results/tax+"
df_test = {}
if os.path.exists(workspace):
    for fname in os.listdir(workspace):
        if fname[0] == '.':
            continue
        cont = fname[:-9].split('_')
        ds = cont[0]
        sel = "_".join(cont[1:])
        if sel not in df_test:
            df_test[sel] = {x:0 for x in ["ds12", "ds3", "mix", "dds3"]}
        df_tmp = pd.DataFrame(pd.read_csv(f"{workspace}/{fname}", index_col=None, header=None))
        df_test[sel][ds] = computeFromPROB(df_tmp)
else:
    print(f"Warning: Directory {workspace} does not exist. Skipping...")
    

In [ ]:
workspace = "../data/results"
for sel in ['tax14']:
    df_test[sel] = {}
    for ds in ["ds12", "ds3", "mix", "dds3"]:
        df_tmp = pd.DataFrame(pd.read_csv(f"{workspace}/{ds}_{sel}_prob.csv", index_col=None, header=None))
        df_test[sel][ds] = computeFromPROB(df_tmp)



In [ ]:
print("    \t[12|12]\t[3|3]\t[123|123] [123|3]\t[12|12]\t[3|3]\t[123|123] [123|3]")
for sel in df_test:
    print("{}\t{:.4f}\t{:.4f}\t{:.4f}\t  {:.4f}\t{:.4f}\t{:.4f}\t{:.4f}\t  {:.4f}".format(
            sel,
            df_test[sel]['ds12']['MCC'].mean(),
            df_test[sel]['ds3']['MCC'].mean(),
            df_test[sel]['mix']['MCC'].mean(),
            df_test[sel]['dds3']['MCC'].mean(),
            df_test[sel]['ds12']['macro_f1'].mean(),
            df_test[sel]['ds3']['macro_f1'].mean(),
            df_test[sel]['mix']['macro_f1'].mean(),
            df_test[sel]['dds3']['macro_f1'].mean())
         )
   

## all pathway

In [ ]:
workspace = "../data/results/pathway"
df_test = {}
if os.path.exists(workspace):
    for fname in os.listdir(workspace):
        if fname[0] == '.':
            continue
        cont = fname[:-9].split('_')
        ds = cont[0]
        sel = "_".join(cont[1:])
        if sel not in df_test:
            df_test[sel] = {x:0 for x in ["ds12", "ds3", "mix", "dds3"]}
        df_tmp = pd.DataFrame(pd.read_csv(f"{workspace}/{fname}", index_col=None, header=None))
        df_test[sel][ds] = computeFromPROB(df_tmp)
else:
    print(f"Warning: Directory {workspace} does not exist. Skipping...")
    

In [ ]:
print("pathway\tR12T12\tR3T3\tR123T123\tR123T3\tR12T12\tR3T3\tR123T123\tR123T3")
for sel in df_test:
    print("{}\t{:.4f}\t{:.4f}\t{:.4f}\t{:.4f}\t{:.4f}\t{:.4f}\t{:.4f}\t{:.4f}".format(
            sel,
            df_test[sel]['ds12']['MCC'].mean(),
            df_test[sel]['ds3']['MCC'].mean(),
            df_test[sel]['mix']['MCC'].mean(),
            df_test[sel]['dds3']['MCC'].mean(),
            df_test[sel]['ds12']['macro_f1'].mean(),
            df_test[sel]['ds3']['macro_f1'].mean(),
            df_test[sel]['mix']['macro_f1'].mean(),
            df_test[sel]['dds3']['macro_f1'].mean())
         )
   

In [ ]:
workspace = "../data/results"
df_test = {}
for fname in os.listdir(workspace):
    if fname[0] == '.':
        continue
    cont = fname[:-9].split('_')
    ds = cont[0]
    sel = "_".join(cont[1:])
    if sel not in df_test:
        df_test[sel] = {x:0 for x in ["ds12", "ds3", "mix", "dds3"]}
    df_tmp = pd.DataFrame(pd.read_csv(f"{workspace}/{fname}", index_col=None, header=None))
    df_test[sel][ds] = computeFromPROB(df_tmp)

## prove one-carbon

In [ ]:
workspace = "../data/results"
df_test = {}
for sel in ['1C_methanogenesis'
    # 'Akkermansia_muciniphila', 
#             'Methanobrevibacter_smithii',
#             "one_carbon", "methanogenesis", "PFS",
#             'tax14', 'kegg47', 'pfam31', 'K30x96',
#             'ds3pfam', 'ds3tax', 
           ]:
    df_test[sel] = {}
    for ds in ["ds12", "ds3", "mix", "dds3"]:
        df_test[sel][ds] = pd.DataFrame(pd.read_csv(f"{workspace}/{ds}_{sel}_prob.csv", index_col=None, header=None))
        
for sel in df_test:
    for ds in ["ds12", "ds3", "mix", "dds3"]:
        df_test[sel][ds] = computeFromPROB(df_test[sel][ds])
df_test

In [ ]:
selectSet = [['1C_methanogenesis']]
    # ['Akkermansia_muciniphila', 'Methanobrevibacter_smithii', "one_carbon", "methanogenesis", "PFS"],
    # ['tax14', 'kegg47', 'pfam31', 'K30x96'],
    # ['ds3tax', 'ds3pfam']]

In [ ]:
# for table
for measure in ["macro_f1", "MCC"]:
    print(measure)
    print("Dataset\tmodel\tguess\tP-value")        
    for sel in selectSet[0]:
        print(sel)
        for ds in df_test[sel]:
            print("{}\t{:.4f}\t{:.4f}\t{:.4f}".format(
                ds, 
                df_test[sel][ds][measure].mean(), 
                df_guess[ds][measure].mean(), 
                computeP_value(df_guess[ds][measure], df_test[sel][ds][measure].mean())))    
        print("")    


In [ ]:
# search go list
pfam_l = ['PF11202', 'PF15609', 'PF07811', 'PF08495', 'PF03845', 'PF12500', 'PF16924', 'PF15608', 'PF13886', 'PF10442', 'PF14464', 'PF16813', 'PF06908', 'PF02424', 'PF12996', 'PF14376', 'PF03646', 'PF11728', 'PF00800', 'PF07690', 'PF09986', 'PF03449', 'PF15644', 'PF02272', 'PF11391', 'PF10138', 'PF02342', 'PF13690', 'PF10060', 'PF13288', 'PF12995', 'PF08460', 'PF00141', 'PF03776', 'PF11171', 'PF11655', 'PF01081', 'PF01339', 'PF10544', 'PF06028', 'PF00188', 'PF03448', 'PF00700', 'PF12939', 'PF12994', 'PF02576', 'PF13936', 'PF14794', 'PF14285', 'PF12686', 'PF16341', 'PF06463', 'PF04316', 'PF00015', 'PF14132', 'PF00334', 'PF12986', 'PF01235', 'PF02737', 'PF07883', 'PF16169', 'PF09347', 'PF16524', 'PF05869', 'PF11363', 'PF02670', 'PF07282', 'PF05119', 'PF00213', 'PF12641', 'PF09148', 'PF13580', 'PF00762', 'PF03616', 'PF05031', 'PF07683', 'PF00034', 'PF14628', 'PF02464', 'PF08930', 'PF00793', 'PF07505', 'PF05035', 'PF15617', 'PF00374', 'PF06619', 'PF13791', 'PF03884', 'PF01706', 'PF00959']

_go_path = "../../database/pfam/gene_ontology.txt"  # external Pfam->GO mapping, not shipped in this repo
if os.path.exists(_go_path):
    df_go = pd.DataFrame(pd.read_csv(_go_path, sep='\t',header=None,index_col=None))
else:
    print(f"Warning: {_go_path} not found; skipping GO lookup."); df_go = pd.DataFrame(columns=[0,1,2,3])
df_go.columns = ['PfamId', 'GOID', 'GO term', 'GO type']

# with open("all_go_term.txt", 'w') as fout:
#     for go_term in df_go['GO term']:
#         fout.write(go_term)
#         fout.write("\t")
#         for pid in df_go[df_go['GO term'] == go_term]['PfamId']:
#             fout.write(pid)
#             fout.write(" ")
#         fout.write("\n")

# df_go[df_go['PfamId'].isin(pfam_l)].to_csv("pfam_ds3_go.csv")



## barplot

In [ ]:
# choose color
measure = "MCC"
clc4 = sns.color_palette([getRGB( 73,148,196),getRGB( 18, 38, 79),getRGB(250,192, 61),getRGB(234, 85, 20)])
sns.palplot(clc4)

color_map = {
    'ds12':getRGB( 73,148,196), 'ds3':getRGB( 18, 38, 79), 
    'mix':getRGB(250,192, 61), 'dds3':getRGB(234, 85, 20)
}
mapping_printName = {
    "Akkermansia_muciniphila":  "A. muciniphila",
    "one_carbon":               "1C metabolic process", # GO:0006730
    "methanogenesis":           "methanogenesis",       # GO:0015948
    "PFS":                      "PFS_Liu_Cancers",
    "ds12": "R12T12",
    "ds3":  "R3T3",
    "mix":  "R123T123",
    "dds3": "R123T3",
}
mapping_ps_ds = {
    'PFS':                     {'ds12':0,  'ds3':1,  'mix':2,  'dds3':3},
    "one_carbon":              {'ds12':6,  'ds3':7,  'mix':8,  'dds3':9},
    "methanogenesis":          {'ds12':12, 'ds3':13, 'mix':14, 'dds3':15},
    "Akkermansia_muciniphila": {'ds12':18, 'ds3':19, 'mix':20, 'dds3':21},
    
}

fsize = (20,10)
fontsize = 20
legend_properties = {'weight':'bold', 'size':fontsize}
patch = []
for ds in color_map:
    patch.append(mpatches.Patch(color=color_map[ds], label=mapping_printName[ds]))


In [ ]:
gm  = [0]*22
std = [0]*22
upperString = [""]*22

for pf in ['Akkermansia_muciniphila', "one_carbon", "methanogenesis", "PFS"]:
    for ds in ['ds12', 'ds3', 'mix', 'dds3']:
        gm[mapping_ps_ds[pf][ds]]  = df_test[pf][ds][measure].mean()
        std[mapping_ps_ds[pf][ds]] = df_test[pf][ds][measure].std()
        
        pv = computeP_value(df_guess[ds][measure], df_test[pf][ds][measure].mean())
        val = f"{df_test[pf][ds][measure].mean():.4f}"
        if pv >= 0.05:
            upperString[mapping_ps_ds[pf][ds]] = val
        elif pv >= 0.005:
            upperString[mapping_ps_ds[pf][ds]] = val+'*'
        elif pv >= 0.0005:
            upperString[mapping_ps_ds[pf][ds]] = val+'**'
        elif pv >= 1e-5:
            upperString[mapping_ps_ds[pf][ds]] = val+'***'
        else:
            upperString[mapping_ps_ds[pf][ds]] = val+'****'   # P < 1e-5


In [ ]:
sns.set(font="Arial")
sns.set_style("white")
fig, ax = plt.subplots(1,1,figsize=fsize)

for pf in ['Akkermansia_muciniphila', "one_carbon", "methanogenesis", "PFS"]:
    for ds in ['ds12', 'ds3', 'mix', 'dds3']:
        i = mapping_ps_ds[pf][ds]
        ax.bar(i, gm[i], edgecolor=color_map[ds], color=color_map[ds], linewidth=3, alpha=0.7)
        ax.plot([i-0.2,i+0.2], [gm[i]-std[i],gm[i]-std[i]],c=color_map[ds],linewidth=3)
        ax.plot([i,i],         [gm[i]-std[i],gm[i]+std[i]],c=color_map[ds],linewidth=3)
        ax.plot([i-0.2,i+0.2], [gm[i]+std[i],gm[i]+std[i]],c=color_map[ds],linewidth=3)
        
ax.legend(handles=patch, loc="topleft", prop=legend_properties, shadow=False, frameon=False) 

ax.text(0.1, gm[0]+0.01, upperString[0], legend_properties, color=color_map['ds12'])
ax.text(0.5, gm[1]+0.07, upperString[1], legend_properties, color=color_map['ds3'])
ax.text(1.5, gm[2]+0.07, upperString[2], legend_properties, color=color_map['mix'])
ax.text(3,   gm[3]+0.01, upperString[3], legend_properties, color=color_map['dds3'])

ax.text(6.1, gm[6]+0.01, upperString[6], legend_properties, color=color_map['ds12'])
ax.text(6.5, gm[7]+0.08, upperString[7], legend_properties, color=color_map['ds3'])
ax.text(7.5, gm[8]+0.05, upperString[8], legend_properties, color=color_map['mix'])
ax.text(9.1, gm[9]+0.01, upperString[9], legend_properties, color=color_map['dds3'])

ax.text(12,   gm[12]+0.01,  upperString[12], legend_properties, color=color_map['ds12'])
ax.text(12.5, gm[13]+0.06,  upperString[13], legend_properties, color=color_map['ds3'])
ax.text(13.5, gm[14]+0.05,  upperString[14], legend_properties, color=color_map['mix'])
ax.text(15.1, gm[15]+0.005, upperString[15], legend_properties, color=color_map['dds3'])

ax.text(16.5, gm[18]+0.01, upperString[18], legend_properties, color=color_map['ds12'])
ax.text(18.3, gm[19]+0.07, upperString[19], legend_properties, color=color_map['ds3'])
ax.text(19.3, gm[20]+0.08, upperString[20], legend_properties, color=color_map['mix'])
ax.text(20.3, gm[21]+0.06, upperString[21], legend_properties, color=color_map['dds3'])

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.text(-0.25, -0.025, mapping_printName["PFS"], legend_properties)
ax.text(5.25, -0.025, mapping_printName["one_carbon"], legend_properties)
ax.text(11.75, -0.025, mapping_printName["methanogenesis"], legend_properties)
ax.text(18, -0.025, mapping_printName["Akkermansia_muciniphila"], legend_properties)

plt.savefig("barplot.pdf", bbox_inches='tight', dpi = 1000)


In [ ]:
def plotAntiRBD(df_data, timepoints, name):
    gm = np.zeros((len(timepoints)))
    c1 = np.zeros((len(timepoints)))
    c2 = np.zeros((len(timepoints)))
    for (j,t) in enumerate(timepoints):
            x = df_data[(df_data["time"] == t)]['Result (U/mL)'].dropna()
            y = np.log(x) 
            gm[j] = stats.gmean(x)
            c1[j] = np.exp(stats.norm.interval(alpha=0.95, loc=np.mean(y), scale=stats.sem(y))[0])
            c2[j] = np.exp(stats.norm.interval(alpha=0.95, loc=np.mean(y), scale=stats.sem(y))[1])

    sns.set(font="Arial")
    sns.set_style("white")
    fig, ax = plt.subplots(1,1,figsize=fsize)
    
    ax.bar(np.arange(len(timepoints)), gm, edgecolor=box_color, color='None', linewidth=2)
    sns.stripplot(x="time", y="Result (U/mL)", 
                  data=df_data, color=RBD_color,
                  order=timepoints,
                  ax=ax)

    for i in range(len(timepoints)):
        sns.lineplot(x=[i-0.2,i+0.2], y=c1[i],         color=box_color, linewidth = 3, ax=ax)
        ax.plot(       [i,i], [c1[i],c2[i]],           c=box_color,     linewidth = 3)
        sns.lineplot(x=[i-0.2,i+0.2], y=c2[i],         color=box_color, linewidth = 3, ax=ax)


    sns.lineplot(x=[-0.8,len(timepoints)-0.2], y=[0.8,0.8], color='k', linestyle="dashed", ax=ax)
    sns.lineplot(x=[-0.8,len(timepoints)-0.2], y=[100,100], color='k', linestyle="dashed", ax=ax)

    ax.set_yscale("log")
    ax.set_xlabel('')
    ax.set_ylabel('Anti-RBD Antibody (U/ml)', size=fontsize, weight="bold")
    ax.set_xlim(-0.8,len(timepoints)-0.2)

    ticklabel = []
    for it in ax.get_yticks():
        ticklabel.append(str(it))
    ax.set_yticklabels(ticklabel, fontdict={'fontsize': fontsize, 'fontweight':'bold'})
    ax.set_yscale("log")

    ticklabel = [tp_map[k] for k in timepoints]
    ax.set_xticklabels(ticklabel, fontdict={'fontsize': fontsize, 'fontweight':'bold'})


    rows = ["GMT"]
    tt = []
    for it in gm:
        tt.append(str(np.round(it, 2)))
    cell_text = [tt]  
    the_table = ax.table(cellText=cell_text,
                     rowLabels=rows,
                     cellLoc="center",
                     loc='top',
                     edges='open')
    for (row, col), cell in the_table.get_celld().items():
        cell.set_text_props(fontproperties=FontProperties(weight='bold', size=fontsize))
    the_table.scale(1, 4)
    
    plt.savefig(name+".pdf", bbox_inches='tight', dpi = 1000)
    plt.savefig(name+".png", bbox_inches='tight', dpi = 1000)
    plt.close()

# Corrected Test

In [ ]:
workspace = "../data/results"
df_test = {}
for sel in ['Akkermansia_muciniphila', 
            'Methanobrevibacter_smithii',
            "one_carbon", "methanogenesis", "PFS"
           ]:
    df_test[sel] = {}
    for ds in ["ds12", "ds3", "mix", "dds3"]:
        df_test[sel][ds] = pd.DataFrame(pd.read_csv(f"{workspace}/{ds}_{sel}.csv", index_col=0))

# print("Dataset\t\tmodel F-socre\tguess F-score\tP-value")        
# for sel in df_test:
#     print(sel)
#     for ds in df_test[sel]:
#         print("{:8}\t{:.4f}\t\t{:.4f}\t\t{:.4f}".format(
#             ds, 
#             df_test[sel][ds]['f1_score'].mean(), 
#             df_guess[ds]['f1_score'].mean(), 
#             computeP_value(df_guess[ds]['f1_score'], df_test[sel][ds]['f1_score'].mean())))    
#     print("")       

In [ ]:
workspace = "../data/results"
df_test = {}
for sel in ['tax14', 'kegg47', 'pfam31', 'K30x96']:
    df_test[sel] = {}
    for ds in ["ds12", "ds3", "mix", "dds3"]:
        df_test[sel][ds] = pd.DataFrame(pd.read_csv(f"{workspace}/{ds}_{sel}.csv", index_col=0))

print("Dataset\t\tmodel F-socre\tguess F-score\tP-value")        
for sel in df_test:
    print(sel)
    for ds in df_test[sel]:
        print("{:8}\t{:.4f}\t\t{:.4f}\t\t{:.4f}".format(
            ds, 
            df_test[sel][ds]['f1_score'].mean(), 
            df_guess[ds]['f1_score'].mean(), 
            computeP_value(df_guess[ds]['f1_score'], df_test[sel][ds]['f1_score'].mean())))    
    print("")       

# Pure Test

# find measuring matrix